In [ ]:
import time
import sys
import threading
from pymata4 import pymata4
import FreeSimpleGUI as sg

In [2]:
board = pymata4.Pymata4(com_port="COM4")

pymata4:  Version 1.15

Copyright (c) 2020 Alan Yorinks All Rights Reserved.

Opening COM4...

Waiting 4 seconds(arduino_wait) for Arduino devices to reset...
Arduino compatible device found and connected to COM4

Retrieving Arduino Firmware ID...
Arduino Firmware ID: 1.2 Reacting_button_press_pymata4_using_timer.ino

Retrieving analog map...
Auto-discovery complete. Found 20 Digital Pins and 6 Analog Pins




In [26]:
LED_PIN = 4
board.set_pin_mode_digital_output(LED_PIN)
board.digital_write(LED_PIN,0)

In [ ]:
board.digital_write(LED_PIN,1)

In [ ]:
# Quick raw test: is the button on D6 reporting anything at all?
BUTTON_PIN = 6

def test_button_callback(data):
    print(f"button event: {data}")

board.set_pin_mode_digital_input(BUTTON_PIN, callback=test_button_callback)
print("Watching D6 - press the button and look for printed events above.")

Watching D6 - press the button and look for printed events above.


button event: [0, 6, 1, 1784033355.5207462]
button event: [0, 6, 0, 1784033355.7264526]
button event: [0, 6, 1, 1784033356.487843]
button event: [0, 6, 0, 1784033356.6129982]
button event: [0, 6, 1, 1784033357.6933932]
button event: [0, 6, 0, 1784033357.824518]
button event: [0, 6, 1, 1784033528.9597957]
button event: [0, 6, 0, 1784033529.0801408]


In [ ]:
# Button -> LED timer logic (README items 3 & 5)
BUTTON_PIN = 6
LED_PIN = 4
board.set_pin_mode_digital_input(BUTTON_PIN)

LED_ON_MS = 1000  # LED ON duration in ms; the GUI cell below can change this live

led_off_timer = None       # holds the pending "turn off" timer so a new press can replace it
last_button_state = "unknown"
last_action = "-"

def turn_off_led():
    """threading.Timer callback: runs once LED_ON_MS has elapsed since the press."""
    global last_action
    board.digital_write(LED_PIN, 0)
    last_action = "LED off (timer expired)"
    print(last_action)

def button_callback(data):
    """pymata4 calls this on its own thread whenever BUTTON_PIN changes state.
    data = [pin_mode, pin_number, value, timestamp]."""
    global led_off_timer, last_button_state, last_action

    print("in callback")

    _, pin_number, value, _ = data
    last_button_state = "pressed" if value == 1 else "released"
    if value != 1:
        return  # only react to the press edge, not the release

    board.digital_write(LED_PIN, 1)
    last_action = f"LED on for {LED_ON_MS} ms"
    print(f"Button pressed on pin {pin_number} -> {last_action}")

    if led_off_timer is not None:
        led_off_timer.cancel()  # a new press restarts the timer instead of stacking timers
    led_off_timer = threading.Timer(LED_ON_MS / 1000, turn_off_led)
    led_off_timer.start()

board.set_pin_mode_digital_input(BUTTON_PIN, callback=button_callback)
print("Button callback registered. Press the button to test.")

In [9]:
# GUI (README item 6): shows button state + last action, and lets you change the
# LED ON duration live. button_callback (previous cell) keeps running on pymata4's
# own thread while this event loop runs, so a button press updates the GUI in real time.
layout = [
    [sg.Text("LED ON duration (ms):"),
     sg.Input(str(LED_ON_MS), key="-MS-", size=(10, 1)),
     sg.Button("Apply")],
    [sg.Text("Button state:"), sg.Text(last_button_state, key="-STATE-", size=(12, 1), font=("Helvetica", 10, "bold"))],
    [sg.Text("Last action:"), sg.Text(last_action, key="-ACTION-", size=(30, 1))],
    [sg.Button("Close")],
]
window = sg.Window("Button -> LED Timer (pymata4)", layout, finalize=True)

while True:
    event, values = window.read(timeout=200)  # poll every 200ms to refresh the display
    if event in (sg.WINDOW_CLOSED, "Close"):
        break

    if event == "Apply":
        try:
            new_ms = int(values["-MS-"])
            if new_ms > 0:
                LED_ON_MS = new_ms
            else:
                sg.popup_error("Enter a positive number of milliseconds.")
        except ValueError:
            sg.popup_error("Enter a whole number of milliseconds.")

    window["-STATE-"].update(last_button_state)
    window["-ACTION-"].update(last_action)

window.close()

NameError: name 'sg' is not defined

In [ ]:
# board.shutdown() crash!